# Object Break - Interactive Demo

## Try It Yourself! Break objects into pieces and watch them fly apart.

This notebook lets you experiment with all the parameters:
- **Shape**: sphere, box, cylinder, cone
- **Number of pieces**: how many fragments
- **Explosion mode**: how the pieces fly apart
- **Force, velocity, gravity**: physics controls
- **Trajectory trails**: visualize the path of each piece

### Setup
```bash
pip install -e .
```

In [ ]:
import numpy as np
import trimesh
from pathlib import Path
from IPython.display import Video, display, HTML

from object_break.fracture.seeds import impact_biased_seeds, random_surface_point
from object_break.fracture.cutter import fracture_mesh
from object_break.physics.simulation import FragmentSimulation, EXPLOSION_MODES
from object_break.render.renderer import SequenceRenderer

## Helper function

This wraps the full pipeline: create shape -> fracture -> simulate -> render -> display video.

In [ ]:
def break_object(
    shape="sphere",       # sphere, box, cylinder, cone
    radius=1.0,           # size of the object
    height=1.5,           # height (cylinder/cone only)
    pieces=3,             # number of fragments
    mode="explode",       # explode, impact, split_half, scatter, peel, directional
    force=3.0,            # impact force
    velocity=1.5,         # velocity multiplier
    gravity=(0, 0, 0),    # gravity vector (try (0,0,-5) for falling)
    duration=3.0,         # video length in seconds
    hold=0.2,             # seconds showing intact object
    trails=True,          # show trajectory trails
    fps=30,               # frames per second
    resolution=512,       # image resolution
    seed=42,              # random seed for reproducibility
    output_dir="output",  # where to save
):
    """Break an object and return the video path."""
    rng = np.random.default_rng(seed)
    
    # Create shape
    shapes = {
        "sphere": lambda: trimesh.creation.icosphere(subdivisions=3, radius=radius),
        "box": lambda: trimesh.creation.box(extents=[radius*2, radius*2, radius*2]),
        "cylinder": lambda: trimesh.creation.cylinder(radius=radius, height=height),
        "cone": lambda: trimesh.creation.cone(radius=radius, height=height),
    }
    mesh = shapes[shape]()
    
    # Fracture
    impact_point, impact_normal = random_surface_point(mesh, rng=rng)
    impact_direction = -impact_normal
    seeds = impact_biased_seeds(mesh, impact_point, pieces, spread=2.0, rng=rng)
    fragments = fracture_mesh(mesh, seeds)
    print(f"{shape} -> {len(fragments)} fragments")
    
    # Simulate
    hold_frames = int(hold * fps)
    sim_frames = int(duration * fps) - hold_frames
    
    sim = FragmentSimulation(
        fragments=fragments,
        mode=mode,
        impact_point=impact_point,
        impact_direction=impact_direction,
        force=force,
        gravity=np.array(gravity),
        velocity_scale=velocity,
        angular_velocity_scale=2.0,
        damping=0.005,
        rng=rng,
    )
    sim_result = sim.run(num_frames=sim_frames, fps=fps, hold_frames=hold_frames)
    
    # Render
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    
    renderer = SequenceRenderer(
        resolution=(resolution, resolution),
        camera_distance=radius * 5.0,
        camera_elevation=25.0,
        camera_azimuth=45.0,
        show_trails=trails,
        trail_length=int(fps * 0.5),
    )
    video_path = renderer.render_sequence(
        fragments=fragments,
        sim_result=sim_result,
        output_dir=out,
        intact_mesh=mesh,
        save_video=True,
        save_frames=False,
    )
    return video_path

---
## 1. Explode Mode

All pieces fly **outward from the center** in every direction. Classic explosion.

In [ ]:
v = break_object(mode="explode", pieces=3, output_dir="output/explode")
Video(str(v), embed=True, width=512)

---
## 2. Impact Mode

Simulates a **hit from one side**. Pieces near the impact fly faster, pieces far away move slower. Directional bias along the impact vector.

In [ ]:
v = break_object(mode="impact", pieces=3, output_dir="output/impact")
Video(str(v), embed=True, width=512)

---
## 3. Split Half Mode

Object **splits along a random axis** into two groups that separate in opposite directions. Like cracking in two.

In [ ]:
v = break_object(mode="split_half", pieces=3, output_dir="output/split_half")
Video(str(v), embed=True, width=512)

---
## 4. Scatter Mode

Each piece gets a **completely random direction**. Chaotic, unpredictable breakage.

In [ ]:
v = break_object(mode="scatter", pieces=3, output_dir="output/scatter")
Video(str(v), embed=True, width=512)

---
## 5. Peel Mode

Pieces **peel off the surface** with a tangential component, like a shell coming apart. More graceful motion.

In [ ]:
v = break_object(mode="peel", pieces=3, output_dir="output/peel")
Video(str(v), embed=True, width=512)

---
## 6. Directional Mode

All pieces fly in **roughly the same direction** with some spread. Like being hit by a strong wind or shockwave.

In [ ]:
v = break_object(mode="directional", pieces=3, output_dir="output/directional")
Video(str(v), embed=True, width=512)

---
## Play With Parameters!

Try changing these and re-running:

In [ ]:
# More pieces, different shape
v = break_object(
    shape="box",          # try: sphere, box, cylinder, cone
    pieces=8,             # more fragments
    mode="explode",       # try: explode, impact, split_half, scatter, peel, directional
    force=5.0,            # stronger explosion
    velocity=2.0,         # faster pieces
    gravity=(0, 0, -3),   # add some downward gravity
    duration=4.0,         # 4 second video
    trails=True,          # show trajectory lines
    seed=123,             # change seed for different fracture pattern
    output_dir="output/custom",
)
Video(str(v), embed=True, width=512)

---
## Batch: Generate Training Data

For training diffusion models, generate many variations:

In [ ]:
# Generate 6 videos: every explosion mode on the same object
for mode in EXPLOSION_MODES:
    v = break_object(
        shape="sphere",
        pieces=5,
        mode=mode,
        duration=3.0,
        output_dir=f"output/batch/{mode}",
        seed=42,
    )
    print(f"  {mode}: {v}")

In [ ]:
# Generate variations with different seeds
for i in range(5):
    v = break_object(
        pieces=4,
        mode="explode",
        seed=i,
        output_dir=f"output/variations/seed_{i}",
    )
    print(f"  seed {i}: {v}")